# **SETUP**

In [4]:
import pathlib
import pandas as pd
from sklearn.preprocessing import SplineTransformer

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

num_cols = train.select_dtypes("number").drop(columns=["PitStop", "PitNextLap"]).columns
st = SplineTransformer().set_output(transform="pandas")

# **SPLINE TRANSFORM NUMERICS**

In [5]:
train_subset = train[num_cols]
test_subset = test[num_cols]
oofs = []

for k in sorted(cv.outer_fold.unique()):
    is_val = cv["outer_fold"] == k
    st.fit(train_subset[~is_val])
    oofs.append(st.transform(train_subset[is_val]))

_ = st.fit(train_subset)

# **EXPORT**

In [8]:
feature_name = "007-spline-transform-numerics"
(PROJECT_ROOT / "data" / "features" / feature_name).mkdir(exist_ok=True)

oof_out = pd.concat(oofs).sort_index()
oof_out.to_parquet(PROJECT_ROOT / "data" / "features" / feature_name / "train.parquet")

test_out = st.transform(test_subset)
test_out.to_parquet(PROJECT_ROOT / "data" / "features" / feature_name / "test.parquet")